In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

In [ ]:
# External dataset is intentionally omitted from the repository.
DATA_ROOT = "path/to/fake news recognition"
true_df = pd.read_csv(f"{DATA_ROOT}/true.csv")
fake_df = pd.read_csv(f"{DATA_ROOT}/fake.csv")


In [ ]:
fake_df = pd.read_csv("path/to/external/data")
fake_df

## **1.?곗씠??遺꾩꽍?섍린**

### **1-1. 二쇱젣蹂??댁뒪 ??遺꾩꽍**

In [ ]:
import matplotlib.pyplot as plt

true_news_count = true_df['subject'].value_counts()
fake_news_count = fake_df['subject'].value_counts()

plt.figure(figsize=(12, 6))
plt.bar(true_news_count.index, true_news_count.values, color='blue', alpha=0.7, label='True')
plt.bar(fake_news_count.index, fake_news_count.values, color='red', alpha=0.7, label='Fake')

plt.title('# of News Articles')
plt.xlabel('Subject')
plt.ylabel('# of News Articles')
plt.xticks(rotation=45)
plt.legend()
plt.show()

### **1-2. ?쒓컙蹂??댁뒪 ??遺꾩꽍**

In [ ]:
true_df['date'] = pd.to_datetime(true_df['date'], errors='coerce')
fake_df['date'] = pd.to_datetime(fake_df['date'], errors='coerce')

true_news_count_t = true_df['date'].dt.to_period('M').value_counts().sort_index()
fake_news_count_t = fake_df['date'].dt.to_period('M').value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(true_news_count_t.index.to_timestamp(), true_news_count_t.values, marker='o', color='blue', label='True')
plt.plot(fake_news_count_t.index.to_timestamp(), fake_news_count_t.values, marker='o', color='red', label='Fake')

plt.title('# of News Articles by time')
plt.xlabel('time')
plt.ylabel('# of News Articles')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()

### **1-3. 媛??留롮씠 ?깆옣?섎뒗 ?⑥뼱 遺꾩꽍**

##### **1) WordCloud濡??쒓컖??*

In [ ]:
from collections import Counter
from wordcloud import WordCloud

true_text = ' '.join(true_df['text'].values)
fake_text = ' '.join(fake_df['text'].values)

true_word_counts = Counter(true_text.split())
fake_word_counts = Counter(fake_text.split())
top_true_words = dict(true_word_counts.most_common(20))
top_fake_words = dict(fake_word_counts.most_common(20))

wordcloud = WordCloud(width=800, height=400, background_color='white',colormap='cividis').generate_from_frequencies(top_true_words)
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('True News - Top 20 words')
plt.axis('off')
plt.show()

wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='inferno').generate_from_frequencies(top_fake_words)
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Fake News - Top 20 words')
plt.axis('off')
plt.show()

**the of and .... -> 臾댁쓽誘명븳 ?⑥뼱?대?濡??대윭??stopwords瑜??쒓굅?섍퀬 肄붾뱶瑜??ㅼ떆 ?뚮젮蹂댁옄!**

##### **2) WordCloud濡??쒓컖??-> stoprwords ?쒓굅**

In [ ]:
import nltk
from nltk.corpus import stopwords

# NLTK??遺덉슜???ㅼ슫濡쒕뱶
nltk.download('stopwords')

# ?곸뼱 遺덉슜??紐⑸줉 濡쒕뱶
stop_words = set(stopwords.words('english'))

# 遺덉슜???쒓굅
true_text_out = ' '.join(word for word in true_text.split() if word.lower() not in stop_words)
fake_text_out = ' '.join(word for word in fake_text.split() if word.lower() not in stop_words)

true_word_counts_out = Counter(true_text_out.split())
fake_word_counts_out = Counter(fake_text_out.split())
top_true_words_out = dict(true_word_counts_out.most_common(20))
top_fake_words_out = dict(fake_word_counts_out.most_common(20))

wordcloud = WordCloud(width=800, height=400, background_color='white',colormap='cividis').generate_from_frequencies(top_true_words_out)
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('True News without stopwords - Top 20 words')
plt.axis('off')
plt.show()

wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='inferno').generate_from_frequencies(top_fake_words_out)
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Fake News without stopwords - Top 20 words')
plt.axis('off')
plt.show()

##### **3) ?먭렇?섑봽濡??쒓컖??*

In [ ]:
top_true_words_out = dict(true_word_counts_out.most_common(10))
top_fake_words_out = dict(fake_word_counts_out.most_common(10))

# True ?댁뒪 ??洹몃옒??nplt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.pie(top_true_words_out.values(), labels=top_true_words_out.keys(), autopct='%1.1f%%')
plt.title('Top 10 Words in True News')

# Fake ?댁뒪 ??洹몃옒??nplt.subplot(1, 2, 2)
plt.pie(top_fake_words_out.values(), labels=top_fake_words_out.keys(), autopct='%1.1f%%')
plt.title('Top 10 Words in Fake News')

plt.show()

##### **4) ?쒕줈 鍮덈룄???뺤씤**

In [ ]:
import pandas as pd

top_true_words_out = dict(true_word_counts_out.most_common(30))
top_fake_words_out = dict(fake_word_counts_out.most_common(30))

# True ?댁뒪 ?곸쐞 30媛??⑥뼱 ??ndf_true = pd.DataFrame(top_true_words_out.items(), columns=['Word', 'Frequency'])

# Fake ?댁뒪 ?곸쐞 30媛??⑥뼱 ??ndf_fake = pd.DataFrame(top_fake_words_out.items(), columns=['Word', 'Frequency'])

print("Top 30 Words in True News:")
print(df_true)
print("\nTop 30 Words in Fake News:")
print(df_fake)

## **2. ?곗씠???⑹튂湲?*

In [ ]:
true_df['Label'] = 'true'
fake_df['Label'] = 'fake'
combined_df=pd.concat([true_df,fake_df])
combined_df=combined_df.reset_index(drop=True)
combined_df

In [ ]:
combined_df["subject"].value_counts()

## **3. Word2Vec瑜??ъ슜??Semantic Network 遺꾩꽍**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import gensim

from gensim.models import Word2Vec
from gensim.models import KeyedVectors



import urllib.request
from lxml import etree
import xml.etree.ElementTree as elemTree


import re

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt')   # NLTK sent_tokenize 瑜??ъ슜?섍린 ?꾪빐???꾩슂???붿씪??n

import time
import datetime


import warnings
warnings.filterwarnings('ignore')



In [ ]:
!pip show gensim

In [ ]:
model = Word2Vec(sentences=[word_tokenize(text) for text in combined_df['text']], vector_size=100, window=5, min_count=5, workers=4, sg=0)

In [ ]:
# wv ?⑥닔瑜??댁슜?섏뿬 ?⑥뼱??踰≫꽣 媛??뺤씤
vector_value = model.wv["President"]
print(vector_value)

# most_similar ?⑥닔瑜??댁슜?섏뿬 ?좎궗?⑥뼱 李얘린
model_result = model.wv.most_similar("President")
print(model_result)

# ?앹꽦??Word2Vec 紐⑤뜽 ??ν븯怨??ъ궗?⑺븯湲?(諛⑸쾿 1)
model.wv.save_word2vec_format('word2vec_model.bin')

# KeyedVectors ?⑥닔瑜??댁슜?섏뿬 ?덈젴??紐⑤뜽??由щ줈??nloaded_model = KeyedVectors.load_word2vec_format("word2vec_model.bin")

# load ??紐⑤뜽 ?뺤씤?섍린
model_result = loaded_model.most_similar("President")
print(model_result)

# 理쒖쥌 紐⑤뜽 ???諛⑸쾿 2
model.save('word2vec.model')

In [ ]:
loaded_model2 = Word2Vec.load('word2vec.model')

model_result = loaded_model2.wv.most_similar("Trump")

print(model_result)

In [ ]:
loaded_model2 = Word2Vec.load('word2vec.model')

model_result = loaded_model2.wv.most_similar("American")

print(model_result)